# Milestone 13 - Report Integration

Milestone 13 integrates the already standalone Report Agent into the main LangGraph workflow.

## Why Integrate Now

The Report Agent was first verified alone in Milestone 12. In this milestone it is connected after Bug Analysis so the full pipeline can produce `final_results.json`, `report_result.json`, an HTML report, and `dashboard_payload`.

## Integrated Workflow

`START -> orchestrator -> repo_analyzer -> rag -> test_planner -> api_testing -> bug_analysis -> report -> END`

## State Handoff

- Bug Analysis Agent writes `bug_results`, `bug_result_path`, and `recommendations`.
- Report Agent reads `project_info`, `test_plan`, `api_results`, `bug_results`, `recommendations`, and artifact paths.
- Report Agent writes `final_results`, `report_html_path`, and `dashboard_payload`.

## Safety Note

Report Agent does not call `target_url` and does not execute repository code.

Target repository: https://github.com/Vitaee/DjangoRestAPI

Target URL: http://localhost:8000

## Part A - Fake Repo Integration With Mocked API Execution

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch
from IPython.display import HTML, display

from test_auto.graph.workflow import run_workflow

def write_file(root, relative_path, content=""):
    path = Path(root) / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    return path

def make_fake_repo(root):
    repo = Path(root) / "fake_report_integration_repo"
    write_file(repo, "README.md", "# Todo API\nJWT authentication protects Todo CRUD API routes.\n")
    write_file(repo, "requirements.txt", "django\ndjangorestframework\ndjangorestframework-simplejwt\npytest\n")
    write_file(repo, "manage.py", "# placeholder\n")
    write_file(repo, "todo/urls.py", "from django.urls import path\nfrom . import views\nurlpatterns = [path(\"api/todos/\", views.todo_list)]\n")
    write_file(repo, "todo/views.py", "def todo_list(request):\n    pass\n")
    write_file(repo, "templates/login.html", "<form>login</form>\n")
    write_file(repo, "tests/test_todo_api.py", "def test_todo_api(): assert True\n")
    return repo

def mocked_api_result(target_url, test_case, auth_token=None, timeout_seconds=5):
    expected = test_case.get("expected_status") or 200
    return {
        "id": str(test_case.get("id") or "API_UNKNOWN"),
        "name": str(test_case.get("name") or "unnamed"),
        "method": str(test_case.get("method") or "GET").upper(),
        "endpoint": str(test_case.get("endpoint") or ""),
        "status": "passed",
        "expected_status": expected,
        "actual_status": expected,
        "duration_ms": 8.0,
        "details": "mocked",
        "evidence": {},
        "assertions": [{"type": "status_code", "passed": True}],
        "error_type": None,
    }

with TemporaryDirectory() as tmp:
    repo = make_fake_repo(tmp)
    initial_state = {
        "repo_path": str(repo),
        "target_url": "http://localhost:8000",
        "user_preferences": {
            "test_types": ["api"],
            "execution_mode": "sequential",
            "focus": "JWT authentication todo CRUD API tests",
            "rag_top_k": 8,
            "planner_use_llm": False,
            "allow_mutating_api_tests": False,
        },
        "errors": [],
        "agent_logs": [],
    }
    with patch("test_auto.agents.api_testing_agent.execute_api_test_case", side_effect=mocked_api_result):
        final_state = run_workflow(initial_state)

compact = {
    "run_id": final_state["run_id"],
    "api_summary": final_state.get("api_results", {}).get("summary"),
    "bug_summary": final_state.get("bug_results", {}).get("summary"),
    "report_kpis": final_state.get("final_results", {}).get("kpis"),
    "dashboard_payload": final_state.get("dashboard_payload"),
    "final_results_path": final_state.get("final_results_path"),
    "report_result_path": final_state.get("report_result_path"),
    "report_html_path": final_state.get("report_html_path"),
}
compact

{'run_id': 'run_20260520T084211Z_b4be7ac4',
 'api_summary': {'total_tests': 2,
  'passed': 2,
  'failed': 0,
  'skipped': 0,
  'errors': 0,
  'pass_rate': 100.0},
 'bug_summary': {'total_anomalies': 0,
  'high': 0,
  'medium': 0,
  'low': 0,
  'info': 0,
  'by_classification': {}},
 'report_kpis': {'total_api_tests': 2,
  'passed': 2,
  'failed': 0,
  'skipped': 0,
  'errors': 0,
  'pass_rate': 100.0,
  'total_ui_tests': 0,
  'ui_passed': 0,
  'ui_failed': 0,
  'ui_skipped': 0,
  'ui_errors': 0,
  'ui_pass_rate': 0.0,
  'screenshot_count': 0,
  'total_anomalies': 0,
  'high_anomalies': 0,
  'medium_anomalies': 0,
  'low_anomalies': 0,
  'info_anomalies': 0,
  'recommendation_count': 0,
  'global_score': 100.0},
 'dashboard_payload': {'run_id': 'run_20260520T084211Z_b4be7ac4',
  'global_score': 100.0,
  'api': {'total_api_tests': 2,
   'passed': 2,
   'failed': 0,
   'skipped': 0,
   'errors': 0,
   'pass_rate': 100.0},
  'ui': {'total_ui_tests': 0,
   'passed': 0,
   'failed': 0,
   's

In [2]:
if final_state.get("report_html_path"):
    display(HTML(Path(final_state["report_html_path"]).read_text(encoding="utf-8")))

## Part B - Optional Real Target App

This requires the Django target app already running at http://localhost:8000. The notebook does not start the app.

In [3]:
# real_state = run_workflow({
#     "repo_url": "https://github.com/Vitaee/DjangoRestAPI",
#     "target_url": "http://localhost:8000",
#     "user_preferences": {
#         "test_types": ["api", "ui"],
#         "execution_mode": "sequential",
#         "focus": "JWT authentication todo CRUD API tests",
#         "planner_use_llm": False,
#         "allow_mutating_api_tests": False,
#     },
#     "errors": [],
#     "agent_logs": [],
# })
# real_state.get("final_results", {}).get("kpis"), real_state.get("report_html_path")

## Part C - Graph Visualization

In [4]:
from test_auto.graph.workflow import build_graph

graph = build_graph()
try:
    display(HTML(graph.get_graph().draw_mermaid_png()))
except Exception:
    try:
        print(graph.get_graph().draw_mermaid())
    except Exception:
        print("START -> orchestrator -> repo_analyzer -> rag -> test_planner -> api_testing -> bug_analysis -> report -> END")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	orchestrator(orchestrator)
	repo_analyzer(repo_analyzer)
	rag(rag)
	test_planner(test_planner)
	api_testing(api_testing)
	ui_testing(ui_testing)
	bug_analysis(bug_analysis)
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ --> orchestrator;
	api_testing -. &nbsp;end&nbsp; .-> __end__;
	api_testing -.-> bug_analysis;
	api_testing -.-> report;
	api_testing -.-> ui_testing;
	bug_analysis -. &nbsp;end&nbsp; .-> __end__;
	bug_analysis -.-> report;
	orchestrator -. &nbsp;end&nbsp; .-> __end__;
	orchestrator -.-> repo_analyzer;
	rag -. &nbsp;end&nbsp; .-> __end__;
	rag -.-> test_planner;
	repo_analyzer -. &nbsp;end&nbsp; .-> __end__;
	repo_analyzer -.-> rag;
	test_planner -. &nbsp;end&nbsp; .-> __end__;
	test_planner -.-> api_testing;
	test_planner -.-> bug_analysis;
	test_planner -.-> report;
	test_planner -.-> ui_testing;
	ui_testing -. &nbsp;end&nbsp; .-> __end__;
	ui_testing -.-> bu